<a href="https://colab.research.google.com/github/shivansh2310/Quantitative-Portfolio-Management/blob/main/Non_Linear_Alpha_(Machine_Learning)_(Chapter_5).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### A. Tree-Based Ensembles (Gradient Boosting)
Isichenko favors tree-based models like XGBoost and LightGBM for tabular financial data.

* Decision Trees split the data based on conditions (e.g., "Is Volatility < 1.5? If yes, check Momentum. If no, short the stock").

* Ensembling (Boosting): A single decision tree will overfit immediately. Gradient Boosting builds hundreds of very "shallow" trees sequentially. Each new tree tries to fix the errors (residuals) made by the previous trees.

### B. Controlling Overfitting in ML

Financial data has a signal-to-noise ratio worse than almost any other ML domain. If you let an XGBoost model grow deep trees (e.g., max_depth = 10), it will memorize every random price tick in your training data.

* The MFE Rule: In StatArb, we heavily restrict tree depth (max_depth = 3 or 4). We want "weak learners" that only capture the broadest, most robust interactions.


## The XGBoost Alpha Engine

In [30]:
import xgboost as xgb
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

In [31]:
universe = {
    'AAPL': 'Tech', 'MSFT': 'Tech', 'NVDA': 'Tech', 'AMD': 'Tech', 'ORCL': 'Tech',
    'JPM': 'Fin', 'BAC': 'Fin', 'GS': 'Fin', 'MS': 'Fin', 'C': 'Fin',
    'XOM': 'Energy', 'CVX': 'Energy', 'COP': 'Energy', 'EOG': 'Energy', 'SLB': 'Energy',
    'JNJ': 'Health', 'UNH': 'Health', 'PFE': 'Health', 'ABBV': 'Health', 'MRK': 'Health'
}
tickers = list(universe.keys())

print("Fetching historical data... (Takes ~10 seconds)")
# Fetch 1 year of daily close prices
prices = yf.download(tickers, period="1y")['Close']
prices = prices[tickers] # Ensure column order

[*****                 10%                       ]  2 of 20 completed

Fetching historical data... (Takes ~10 seconds)


[*********************100%***********************]  20 of 20 completed


In [32]:
# We use .shift(-1) because today's features must predict tomorrow's return
raw_returns = prices.pct_change()
forward_returns = raw_returns.shift(-1)


In [33]:
# Convert to a "Long" format DataFrame (Standard for ML pipelines)
df = forward_returns.unstack().reset_index()
df.columns = ['Ticker', 'Date', 'Fwd_Return']
df = df.dropna()


In [34]:
# Map the sectors
df['Sector'] = df['Ticker'].map(universe)

In [35]:
def neutralize_and_rank(daily_data):
    # Market Neutralization (Subtract cross-sectional mean)
    daily_data['Market_Mean'] = daily_data['Fwd_Return'].mean()
    daily_data['Market_Neutral'] = daily_data['Fwd_Return'] - daily_data['Market_Mean']

    # Sector Neutralization (Subtract sector mean from the market-neutral returns)
    sector_means = daily_data.groupby('Sector')['Market_Neutral'].transform('mean')
    daily_data['Idiosyncratic_Return'] = daily_data['Market_Neutral'] - sector_means

    # Rank Normalization (Scale between 0 and 1 to suppress outliers)
    daily_data['ML_Target'] = daily_data['Idiosyncratic_Return'].rank(pct=True)

    return daily_data

print("Applying Cross-Sectional Neutralization and Rank Normalization...")
# Apply the function group-by-group for every single day in the dataset
ml_dataset = df.groupby('Date', group_keys=False).apply(neutralize_and_rank)

Applying Cross-Sectional Neutralization and Rank Normalization...


In [36]:
# We must sort by Ticker and Date to calculate rolling features correctly
ml_dataset = ml_dataset.sort_values(['Ticker', 'Date'])

In [37]:
# Short-Term Mean Reversion (5-Day Return)
# Hypothesis: High 5-day return means it's overbought and will revert (negative weight expected)
ml_dataset['F_Reversion_5d'] = ml_dataset.groupby('Ticker')['Fwd_Return'].transform(lambda x: x.shift(1).rolling(5).sum())

# Medium-Term Momentum (21-Day Return)
# Hypothesis: High 1-month return means it's trending (positive weight expected)
ml_dataset['F_Momentum_1m'] = ml_dataset.groupby('Ticker')['Fwd_Return'].transform(lambda x: x.shift(1).rolling(21).sum())

# Daily Volatility (21-Day standard deviation)
ml_dataset['F_Volatility'] = ml_dataset.groupby('Ticker')['Fwd_Return'].transform(lambda x: x.shift(1).rolling(21).std())

ml_dataset = ml_dataset.dropna()

print("Cross-Sectional Feature Standardization (Z-Scoring)...")
# Just like targets, features must be standardized cross-sectionally every single day
features = ['F_Reversion_5d', 'F_Momentum_1m', 'F_Volatility']

Cross-Sectional Feature Standardization (Z-Scoring)...


In [38]:
def standardize_features(daily_data):
    for f in features:
        daily_data[f] = (daily_data[f] - daily_data[f].mean()) / (daily_data[f].std() + 1e-8)
    return daily_data

ml_dataset = ml_dataset.groupby('Date', group_keys=False).apply(standardize_features)


In [39]:
print("Initializing the XGBoost Regressor...")
# MFE Constraints for Financial Data:
# 1. max_depth=3 (Prevent memorizing noise)
# 2. learning_rate=0.05 (Learn slowly)
# 3. subsample=0.8 (Train on 80% of data per tree to prevent overfitting)
xgb_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror'
)

Initializing the XGBoost Regressor...


In [40]:
# X is our features, y is our Neutralized Rank Target from Day 1
X = ml_dataset[features]
y = ml_dataset['ML_Target']

In [41]:
print("Training the Non-Linear Ensemble...")
xgb_model.fit(X, y)

Training the Non-Linear Ensemble...


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.8, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=None, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.05, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=100,
             n_jobs=None, num_parallel_tree=None, ...)

In [42]:
print("Extracting Feature Importance...")
# XGBoost tells us which features were most useful for making splits
importance = pd.Series(xgb_model.feature_importances_, index=features).sort_values(ascending=False)

print("\nNON-LINEAR FEATURE IMPORTANCE (Gain):")
print("="*45)
for feature, imp in importance.items():
    print(f"{feature:>15}: {imp*100:.2f}%")
print("="*45)



Extracting Feature Importance...

NON-LINEAR FEATURE IMPORTANCE (Gain):
 F_Reversion_5d: 37.05%
   F_Volatility: 31.86%
  F_Momentum_1m: 31.09%


In [43]:
# Initialize Ridge Regression with a heavy penalty (alpha)
# In statsmodels it's lambda; in sklearn it's alpha
ridge_model = Ridge(alpha=100.0)
ridge_model.fit(X, y)

Ridge(alpha=100.0)

In [45]:
# Generate predictions
ml_dataset['Prediction'] = ridge_model.predict(X)

# Calculate daily IC (Spearman Rank Correlation between Prediction and True Target)
def calculate_daily_ic(daily_data):
    # Only calculate if we have enough variance
    if daily_data['Prediction'].std() < 1e-8:
        return np.nan
    ic, p_val = spearmanr(daily_data['Prediction'], daily_data['ML_Target'])
    return ic

daily_ic = ml_dataset.groupby('Date').apply(calculate_daily_ic).dropna()
mean_ic = daily_ic.mean()

In [46]:
print("\nEvaluating the ML Model (Information Coefficient)...")
# Generate ML predictions
ml_dataset['XGB_Prediction'] = xgb_model.predict(X)

# Calculate daily IC for the XGBoost model
def calculate_xgb_ic(daily_data):
    if daily_data['XGB_Prediction'].std() < 1e-8:
        return np.nan
    ic, p_val = spearmanr(daily_data['XGB_Prediction'], daily_data['ML_Target'])
    return ic

xgb_daily_ic = ml_dataset.groupby('Date').apply(calculate_xgb_ic).dropna()
xgb_mean_ic = xgb_daily_ic.mean()

print(f"Ridge Model IC (Linear):  {mean_ic:.4f}")
print(f"XGBoost Model IC (Trees): {xgb_mean_ic:.4f}")

if xgb_mean_ic > mean_ic:
    print("\nCONCLUSION: Non-linear alpha discovered. The tree interactions outperformed the linear baseline.")
else:
    print("\nCONCLUSION: Linear dominance. The dataset is too noisy or linear for trees to find complex patterns without overfitting.")


Evaluating the ML Model (Information Coefficient)...
Ridge Model IC (Linear):  0.0155
XGBoost Model IC (Trees): 0.2809

CONCLUSION: Non-linear alpha discovered. The tree interactions outperformed the linear baseline.
